# installments_payments: даёт ли таблица заметный прирост?
Нужны файлы в data/raw/: application_train.csv, bureau.csv,
previous_application.csv, installments_payments.csv

Критерий "оставляем в базовом минимуме": прирост AUC положителен на ВСЕХ фолдах
(min diff > 0) и средний прирост хотя бы порядка 0.003.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, "..")
from src.features import ID_COL, TARGET_COL, build_dataset
from src.models import cv_auc, make_lgbm, make_logreg, to_lgbm

DATA = Path("../data/raw")

In [ ]:
app = pd.read_csv(DATA / "application_train.csv")
bureau = pd.read_csv(DATA / "bureau.csv")
prev = pd.read_csv(DATA / "previous_application.csv")

# installments_payments: ~13.6 млн строк. Читаем только нужные колонки и экономим память.
inst = pd.read_csv(
    DATA / "installments_payments.csv",
    usecols=["SK_ID_PREV", "SK_ID_CURR", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
             "AMT_INSTALMENT", "AMT_PAYMENT"],
    dtype={"SK_ID_PREV": "int32", "SK_ID_CURR": "int32",
           "DAYS_INSTALMENT": "float32", "DAYS_ENTRY_PAYMENT": "float32",
           "AMT_INSTALMENT": "float32", "AMT_PAYMENT": "float32"},
)
print(inst.shape, f"| {inst.memory_usage(deep=True).sum() / 1e6:.0f} MB")

## 1. Что в таблице?

In [ ]:
print(f"Клиентов из app с историей платежей: {app[ID_COL].isin(inst[ID_COL]).mean():.1%}")
print("\nПропуски:")
print(inst.isna().mean().round(5))

In [ ]:
# Просрочка в днях и недоплаты
dpd = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).clip(lower=0)
print(f"Платежей с просрочкой: {(dpd > 0).mean():.1%}")
print(dpd[dpd > 0].describe().round(1))
print(f"\nПлатежей с недоплатой: {((inst['AMT_INSTALMENT'] - inst['AMT_PAYMENT']) > 0).mean():.1%}")

## 2. Собираем два датасета: без installments и с ними

In [ ]:
without = build_dataset(app, bureau, prev)
with_inst = build_dataset(app, bureau, prev, installments=inst)
print("without:", without.shape, "| with:", with_inst.shape)
print("Строк не потеряно:", len(with_inst) == len(app))

In [ ]:
new_cols = [c for c in with_inst.columns if c not in without.columns]
print(f"Новых признаков: {len(new_cols)}")

# дефолт по флагу и по доле просроченных платежей
print(with_inst.groupby("HAS_INST")[TARGET_COL].agg(["mean", "count"]).round(4))

bins = pd.cut(with_inst["INST_IS_LATE_MEAN"], [-0.001, 0, 0.05, 0.15, 0.3, 1.0])
print(with_inst.groupby(bins, observed=True)[TARGET_COL].agg(["mean", "count"]).round(4))

In [ ]:
corr = with_inst[new_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(15).round(3))

## 3. Сравнение на одних и тех же фолдах

In [ ]:
y = with_inst[TARGET_COL]


def prepare(df):
    return df.drop(columns=[TARGET_COL, ID_COL])


X_without, X_with = prepare(without), prepare(with_inst)

In [ ]:
# LightGBM
s_gb_without = cv_auc(make_lgbm(), to_lgbm(X_without), y)
s_gb_with = cv_auc(make_lgbm(), to_lgbm(X_with), y)

In [ ]:
# Логрег
s_lr_without = cv_auc(make_logreg(X_without), X_without, y)
s_lr_with = cv_auc(make_logreg(X_with), X_with, y)

In [ ]:
rows = {}
for name, a, b in [
    ("LightGBM", s_gb_without, s_gb_with),
    ("LogReg", s_lr_without, s_lr_with),
]:
    d = b - a
    rows[name] = {
        "AUC without": a.mean(), "AUC with": b.mean(),
        "mean diff": d.mean(), "min diff": d.min(), "max diff": d.max(),
    }
pd.DataFrame(rows).T.round(4)

## 4. Что реально использует модель
Важность признаков по gain: попали ли INST_* в топ?

In [ ]:
Xl = to_lgbm(X_with)
model = make_lgbm(importance_type="gain").fit(Xl, y)
imp = pd.Series(model.feature_importances_, index=Xl.columns).sort_values(ascending=False)
imp_share = (imp / imp.sum() * 100).round(2)

print("Топ-20 признаков по gain (% от общего):")
print(imp_share.head(20))
print(f"\nДоля gain у INST_*: {imp_share[imp_share.index.str.startswith('INST_')].sum():.1f}%")
print(f"INST_* в топ-30: {imp_share.head(30).index.str.startswith('INST_').sum()}")

## 5. Сохраняем собранный датасет (чтобы не пересчитывать агрегаты)
Нужен pyarrow: pip install pyarrow

In [ ]:
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
with_inst.to_parquet(out / "train_features.parquet", index=False)
print("Сохранено:", out / "train_features.parquet")